<a href="https://colab.research.google.com/github/bainiao0706/Google-Drive-Remote-Upload/blob/main/GoogleDriveRemoteUpload.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#谷歌云盘的离线上传
#作者:bainiao0706

# 挂载Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 离线上传文件

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import os
from urllib.parse import urlparse, unquote
import threading

# 定义目标下载目录
target_dir = '/content/drive/MyDrive/ColabDownloads' # 配置下载目录，默认下载会直接创建一个ColabDownloads的文件夹，编辑格式:/content/drive/MyDrive/你的文件夹名字
os.makedirs(target_dir, exist_ok=True)

# 下载任务列表
# 如果 'original_filename' 为空，则会从下载链接自动命名
download_tasks = [
    {
        "url": "",
        "original_filename": ""
    },
    {
        "url": "",
        "original_filename": ""
    },
    {
        "url": "",
        "original_filename": ""
    }
    # 可以添加更多下载任务

]

# 单个下载函数
def download_file(task):
    url = task['url']
    original_filename = task.get('original_filename', '')

    if not url:
        print(f"Skipping task: URL is empty.")
        return f"FAILED: URL is empty for task {task}"

    # 如果原始文件名为空，则从 URL 中提取
    if not original_filename:
        parsed_url = urlparse(url)
        filename_from_url = os.path.basename(parsed_url.path)
        original_filename = unquote(filename_from_url)
        if not original_filename:
            original_filename = "downloaded_file.bin" # Fallback if no filename in URL
        print(f"Inferred filename for {url}: {original_filename}")

    # 使用线程ID来创建临时的唯一文件名，防止不同线程同时下载时文件名冲突
    temp_filename = f"temp_{threading.get_ident()}_{os.path.basename(original_filename)}.part"
    temp_path = os.path.join(target_dir, temp_filename)
    final_path = os.path.join(target_dir, original_filename)

    print(f"Starting download: {url} -> {final_path}")

    try:
        with requests.get(url, stream=True, allow_redirects=True, timeout=30) as r:
            r.raise_for_status() # 检查 HTTP 请求是否成功
            with open(temp_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

        # 确保下载完整后再重命名
        os.rename(temp_path, final_path)
        print(f"Successfully downloaded and saved: {final_path}")
        return f"SUCCESS: {final_path}"

    except requests.exceptions.RequestException as e:
        print(f"Error downloading {url}: {e}")
        if os.path.exists(temp_path):
            os.remove(temp_path) # 清理部分下载文件
        return f"FAILED: {url} - {e}"
    except Exception as e:
        print(f"An unexpected error occurred for {url}: {e}")
        if os.path.exists(temp_path):
            os.remove(temp_path) # 清理部分下载文件
        return f"FAILED: {url} - {e}"

# 使用 ThreadPoolExecutor 管理并发下载
MAX_CONCURRENT_DOWNLOADS = 5
results = []

with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_DOWNLOADS) as executor:
    # 提交所有下载任务
    future_to_url = {executor.submit(download_file, task): task['url'] for task in download_tasks}

    for future in as_completed(future_to_url):
        url = future_to_url[future]
        try:
            result = future.result()
            results.append(result)
        except Exception as exc:
            results.append(f"FAILED processing {url}: {exc}")

print("\n--- Download Summary ---")
for res in results:
    print(res)
print("------------------------")


In [ ]:
from IPython.display import Javascript
Javascript('window.open("https://github.com/bainiao0706/Google-Drive-Remote-Upload");')